# Multi-Profile Chemical Equilibrium: ExoGibbs vs FastChem vs Emulator

Compare three chemistry backends on the same selected exported model:

1. **ExoGibbs** — differentiable classical reference.
2. **Live FastChem** — subprocess rerun with the same abundance-writing logic
   used during dataset generation.
3. **ML emulator** — the exported bundle selected by `MODEL`.

**Apples-to-apples contract.** By default this notebook builds the ExoGibbs
28-element vector in `fastchem_proxy` mode: the five free elemental globals
(`He_H`, `C_H`, `O_H`, `N_H`, `S_H`) are passed through directly, and the same
hidden-metallicity proxy used during FastChem data generation is applied to the
refractory metals. Set `EXOGIBBS_ELEMENT_MODE = "aas_fixed"` if you want the
older comparison against a fully solar AAG21 background instead.

Scope. This notebook is forward-only. It does not exercise gradients or run a
retrieval. For the gradient checks that qualify the emulator for HMC-NUTS, see
`gradient_verification.ipynb`. For the full ExoJAX retrieval notebook, see
`comparison.ipynb`.


In [ ]:
import datetime
import importlib
import inspect
import json
import sys
import time
from pathlib import Path

import h5py
import numpy as np
import jax.numpy as jnp
from jax import config
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

config.update("jax_enable_x64", True)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "exojax_demo":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL = "fastchem_analytic_500k"
EXOGIBBS_ELEMENT_MODE = "fastchem_proxy"  # or "aas_fixed"
BUNDLE_PATH = PROJECT_ROOT / "models" / MODEL / "best_exported.npz"

# The bundle is weights + metadata; the forward pass lives in
# src/models/transformer.py. ``src/`` ships as a sibling of ``models/`` at
# the distribution root, so BUNDLE_PATH.parents[2] is the dist root and goes
# on sys.path to expose the inference code that matches the weights.
DIST_ROOT = BUNDLE_PATH.resolve().parents[2]
assert (DIST_ROOT / "src").is_dir(), (
    f"src not found at {DIST_ROOT / 'src'} — the distribution must keep "
    "`src/` next to `models/`."
)
sys.path.insert(0, str(DIST_ROOT))

from src.constants import SOLAR_ABUNDANCES
import src.models.classical_reference as classical_reference

classical_reference = importlib.reload(classical_reference)
build_exogibbs_element_vector = classical_reference.build_exogibbs_element_vector
build_exogibbs_species_indices = classical_reference.build_exogibbs_species_indices
mean_abs_log10_error = classical_reference.mean_abs_log10_error
resolve_vulcan_source_root = classical_reference.resolve_vulcan_source_root
run_fastchem_online = classical_reference.run_fastchem_online
from src.models.standalone_inference import load_model


def load_style():
    style_path = PROJECT_ROOT / "extras" / "science.mplstyle"
    if style_path.exists():
        plt.style.use(str(style_path))
    elif Path("science.mplstyle").exists():
        plt.style.use("science.mplstyle")
    else:
        plt.rcParams.update(
            {
                "axes.grid": True,
                "grid.alpha": 0.3,
                "xtick.direction": "in",
                "ytick.direction": "in",
            }
        )


load_style()
bundle = load_model(BUNDLE_PATH)
species_labels = bundle.species
FASTCHEM_SOURCE_ROOT = resolve_vulcan_source_root(bundle.config, project_root=PROJECT_ROOT)
STATE_FLOOR = float(bundle.config.get("preprocessing", {}).get("state_floor", 1.0e-30))
RUN_STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
DEBUG_DIR = PROJECT_ROOT / "exojax_demo" / "diagnostics" / f"exogibbs_vs_emulator_{RUN_STAMP}"
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
print(f"MODEL        : {MODEL}")
print(f"BUNDLE_PATH  : {BUNDLE_PATH}")
print(f"ExoGibbs mode: {EXOGIBBS_ELEMENT_MODE}")
print(f"ML species   : {len(species_labels)}")
print(f"DEBUG_DIR    : {DEBUG_DIR}")
print(f"FC helper    : {inspect.signature(run_fastchem_online)}")



In [ ]:
import jax
print(jax.devices())
print(jax.default_backend())

In [ ]:
from src.models.classical_reference import chemsetup_matched_to_fastchem
from exogibbs.api.equilibrium import EquilibriumOptions, equilibrium_profile
from exojax.utils.zsol import nsol

chem = chemsetup_matched_to_fastchem(FASTCHEM_SOURCE_ROOT)
opts = EquilibriumOptions(epsilon_crit=1e-11, max_iter=1000, method="vmap_cold")
EG_IDX_17 = build_exogibbs_species_indices(chem, species_labels)
solar_exojax = nsol()

# Use the exact training solar anchors for the ML and live-FastChem calls.
# ExoGibbs still uses AAG21 for the 28-element background vector construction.
ML_SOLAR_ABUNDANCES = {key: float(value) for key, value in SOLAR_ABUNDANCES.items()}


def exogibbs_element_vector(globals_map):
    return build_exogibbs_element_vector(
        chem,
        globals_map,
        solar_abundances=solar_exojax,
        mode=EXOGIBBS_ELEMENT_MODE,
    )


print("ExoGibbs configured with", len(chem.elements), "elements.")
print("ML solar abundances (training anchors):", ML_SOLAR_ABUNDANCES)



In [ ]:
from src.data_generation.sampling import (
    _guillot_temperature,
    _apply_upper_atmosphere_modification,
)


def guillot_profile(pressure_bar, *, t_int, t_eq, log10_delta, log10_gamma, alpha, log10_p_trans):
    """Build an in-distribution Piette+Madhusudhan (2019) Guillot profile.

    Parameters are drawn from the same analytic sampler that generated the
    training data (see `src/data_generation/sampling.py`), so the resulting
    T(P) shape lives inside the distribution the emulator learned. Prior
    hand-written power-law profiles (`T = A * p^alpha`) were out-of-
    distribution and triggered large ML↔FC drift via cross-level attention
    (see spec.md "Classical References" / profile-shape sensitivity).
    """
    T = _guillot_temperature(
        pressure_bar,
        delta=10.0 ** log10_delta,
        gamma=10.0 ** log10_gamma,
        t_int_k=t_int,
        t_eq_k=t_eq,
    )
    return _apply_upper_atmosphere_modification(
        pressure_bar, T, alpha=alpha, p_trans_bar=10.0 ** log10_p_trans,
    )


print("Warming up chemistry backends...")
pressures = np.logspace(2, -7, 50)
# Warm-Guillot parameters used for the warm-up pass; identical shape to one
# of the three comparison profiles built in the next cell.
warmup_T = guillot_profile(
    pressures,
    t_int=250.0, t_eq=1500.0,
    log10_delta=-2.0, log10_gamma=-0.3,
    alpha=0.2, log10_p_trans=-2.0,
)
_ = np.asarray(
    bundle.predict_fastchem_profile(
        pressure_bar=pressures,
        temperature_k=warmup_T,
        global_inputs=ML_SOLAR_ABUNDANCES,
        return_log10=False,
    )
)
_ = np.asarray(
    equilibrium_profile(
        chem,
        warmup_T,
        pressures,
        exogibbs_element_vector(ML_SOLAR_ABUNDANCES),
        Pref=1.0,
        options=opts,
    ).x[:, EG_IDX_17]
)
_ = run_fastchem_online(
    FASTCHEM_SOURCE_ROOT,
    pressures,
    warmup_T,
    ML_SOLAR_ABUNDANCES,
    species_labels,
    bundle.config,
)
print("Warmup complete.")


In [ ]:
pressures = np.logspace(2, -7, 50)
# In-distribution Piette+Madhusudhan (2019) Guillot profiles. Parameters
# are drawn from the middle of the training analytic-sampler ranges
# (`config/fastchem_analytic_500k.json`, temperature_profiles.analytic_sampler).
# Each profile's T(P) shape lives inside the distribution the emulator
# learned, so the attention-over-levels context the model sees at inference
# matches what it saw at training. Previous power-law test profiles
# (T = A * p^alpha) were a fundamentally different PT shape and produced
# 0.05–1.3 dex ML↔FC drift driven by profile-shape extrapolation — see
# spec.md "Classical References".
pt_profiles = [
    ("Hot Guillot", guillot_profile(
        pressures,
        t_int=400.0, t_eq=2500.0,
        log10_delta=-2.5, log10_gamma=-0.4,
        alpha=0.3, log10_p_trans=-1.0,
    )),
    ("Warm Guillot", guillot_profile(
        pressures,
        t_int=250.0, t_eq=1500.0,
        log10_delta=-2.0, log10_gamma=-0.3,
        alpha=0.2, log10_p_trans=-2.0,
    )),
    ("Cool Guillot", guillot_profile(
        pressures,
        t_int=150.0, t_eq=800.0,
        log10_delta=-1.5, log10_gamma=-0.3,
        alpha=0.15, log10_p_trans=-2.5,
    )),
]

species_map = [
    ("He", "tab:cyan"),
    ("CO", "tab:blue"),
    ("H2", "tab:orange"),
    ("H2O", "tab:green"),
    ("CH4", "tab:red"),
    ("NH3", "tab:purple"),
]
exogibbs_label = (
    "ExoGibbs (training-matched)"
    if EXOGIBBS_ELEMENT_MODE == "fastchem_proxy"
    else "ExoGibbs (AAS-fixed)"
)

fig, axes = plt.subplots(
    3,
    2,
    figsize=(14, 15),
    sharey="row",
    gridspec_kw={"width_ratios": [1, 2.6], "wspace": 0.0, "hspace": 0.3},
)
profile_results = []

for i, (label, temperatures) in enumerate(pt_profiles):
    t0 = time.time()
    p_ml = np.asarray(
        bundle.predict_fastchem_profile(
            pressure_bar=pressures,
            temperature_k=temperatures,
            global_inputs=ML_SOLAR_ABUNDANCES,
            return_log10=False,
        )
    )
    dt_ml = (time.time() - t0) * 1000.0

    profile_slug = label.lower().replace(" ", "_").replace("-", "_")
    profile_debug_dir = DEBUG_DIR / "fastchem_runtime" / profile_slug
    t0 = time.time()
    p_fc = run_fastchem_online(
        FASTCHEM_SOURCE_ROOT,
        pressures,
        temperatures,
        ML_SOLAR_ABUNDANCES,
        species_labels,
        bundle.config,
        debug_dir=profile_debug_dir,
    )
    dt_fc = (time.time() - t0) * 1000.0

    t0 = time.time()
    p_eg = np.asarray(
        equilibrium_profile(
            chem,
            temperatures,
            pressures,
            exogibbs_element_vector(ML_SOLAR_ABUNDANCES),
            Pref=1.0,
            options=opts,
        ).x[:, EG_IDX_17]
    )
    dt_eg = (time.time() - t0) * 1000.0

    log_ratio_ml_fc = np.log10(np.clip(p_ml, STATE_FLOOR, None)) - np.log10(np.clip(p_fc, STATE_FLOOR, None))
    log_ratio_fc_eg = np.log10(np.clip(p_fc, STATE_FLOOR, None)) - np.log10(np.clip(p_eg, STATE_FLOOR, None))
    log_ratio_ml_eg = np.log10(np.clip(p_ml, STATE_FLOOR, None)) - np.log10(np.clip(p_eg, STATE_FLOOR, None))
    profile_results.append(
        {
            "label": label,
            "pressure_bar": pressures.copy(),
            "temperature_k": np.asarray(temperatures, dtype=np.float64).copy(),
            "ml": p_ml.copy(),
            "fastchem": p_fc.copy(),
            "exogibbs": p_eg.copy(),
            "log10_ratio_ml_fc": log_ratio_ml_fc,
            "log10_ratio_fc_eg": log_ratio_fc_eg,
            "log10_ratio_ml_eg": log_ratio_ml_eg,
            "timings_ms": {
                "ml": float(dt_ml),
                "fastchem": float(dt_fc),
                "exogibbs": float(dt_eg),
            },
            "fastchem_debug_dir": str(profile_debug_dir.resolve()),
        }
    )

    print(
        f"{label:<16} | mean |dex|  ML-FC={mean_abs_log10_error(p_ml, p_fc):.4f}  "
        f"FC-EG={mean_abs_log10_error(p_fc, p_eg):.4f}  "
        f"ML-EG={mean_abs_log10_error(p_ml, p_eg):.4f}"
    )

    ax_pt, ax_vmr = axes[i, 0], axes[i, 1]
    ax_pt.plot(temperatures, pressures, color="black", lw=2)
    ax_pt.set_yscale("log")
    ax_pt.invert_yaxis()
    ax_pt.set_xlim(0, 3000)
    ax_pt.set_xticks([0, 1000, 2000])
    ax_pt.set_ylabel("Pressure (bar)")
    ax_pt.set_xlabel("Temperature (K)")
    ax_pt.set_title(label)

    for species_name, color in species_map:
        species_idx = species_labels.index(species_name)
        ax_vmr.plot(p_eg[:, species_idx], pressures, color=color, ls="-", lw=1, alpha=0.5)
        ax_vmr.plot(p_fc[:, species_idx], pressures, color=color, ls=":", lw=5, alpha=0.5)
        ax_vmr.plot(p_ml[:, species_idx], pressures, color=color, ls="--", lw=5, alpha=0.5)

    ax_vmr.set_xscale("log")
    ax_vmr.set_xlim(1e-10, 3.0)
    ax_vmr.set_xlabel("Mixing Ratio")
    ax_vmr.set_title(
        f"EG {dt_eg:.1f} ms | FC {dt_fc:.1f} ms | ML {dt_ml:.1f} ms"
    )
    ax_vmr.tick_params(axis="y", which="both", left=False, labelleft=False)

    if i == 0:
        legend_handles = [
            Line2D([0], [0], color="k", ls="-", lw=2, alpha=0.7, label=exogibbs_label),
            Line2D([0], [0], color="k", ls=":", lw=2.2, label="Live FastChem"),
            Line2D([0], [0], color="k", ls="--", lw=1.8, label="ML emulator"),
        ]
        legend_handles += [
            Line2D([0], [0], color=color, lw=3, label=species_name)
            for species_name, color in species_map
        ]
        ax_vmr.legend(
            handles=legend_handles,
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            fontsize=10,
        )

plt.show()



## Debug Artifacts

Save the raw backend outputs, pairwise log10-ratio fields, and a coarse
training-coverage audit for the warm power-law corner. This keeps the plot
lightweight while making the failure mode inspectable offline.


In [ ]:
def _slugify(label):
    return label.lower().replace(" ", "_").replace("-", "_")


def _per_species_ratio_stats(log_ratio):
    abs_ratio = np.abs(np.asarray(log_ratio, dtype=np.float64))
    return {
        species: {
            "max_abs_dex": float(np.max(abs_ratio[:, idx])),
            "median_abs_dex": float(np.median(abs_ratio[:, idx])),
        }
        for idx, species in enumerate(species_labels)
    }


def _top_species(log_ratio, pressure_bar, temperature_k, count=5):
    abs_ratio = np.abs(np.asarray(log_ratio, dtype=np.float64))
    order = np.argsort(np.max(abs_ratio, axis=0))[::-1][:count]
    return [
        {
            "species": species_labels[int(idx)],
            "max_abs_dex": float(np.max(abs_ratio[:, idx])),
            "median_abs_dex": float(np.median(abs_ratio[:, idx])),
            "worst_pressure_bar": float(pressure_bar[int(np.argmax(abs_ratio[:, idx]))]),
            "worst_temperature_k": float(temperature_k[int(np.argmax(abs_ratio[:, idx]))]),
        }
        for idx in order
    ]


def _below_floor_counts(values):
    arr = np.asarray(values, dtype=np.float64)
    return {
        species: {
            "below_state_floor": int(np.sum(arr[:, idx] < STATE_FLOOR)),
            "exact_zero": int(np.sum(arr[:, idx] == 0.0)),
        }
        for idx, species in enumerate(species_labels)
    }


def _sample_warm_corner_coverage(sample_limit=2000):
    """Coarse audit of how often the sampled test split visits the Warm Guillot corner.

    This is intentionally approximate. It tells us whether the warm profile lives in
    a densely sampled part of the test manifold once both PT shape and elemental
    abundances are considered together.
    """

    run_ids_path = PROJECT_ROOT / "data" / MODEL / "processed" / "test" / "run_ids.json"
    raw_h5_path = PROJECT_ROOT / "data" / MODEL / "raw" / "runs.h5"
    if not run_ids_path.exists() or not raw_h5_path.exists():
        return {"error": f"Missing test diagnostics inputs under {run_ids_path.parent}."}

    run_ids = json.loads(run_ids_path.read_text())
    stride = max(1, len(run_ids) // sample_limit)
    sampled = run_ids[::stride][:sample_limit]
    p_grid = np.array([1.0e2, 1.0, 1.0e-2, 1.0e-5], dtype=np.float64)
    # Reference T at the four probe pressures matches the Warm Guillot profile in cell 5.
    # Pre-computed from the analytic Piette+Madhusudhan (2019) formulas so the audit
    # does not depend on the Warm Guillot smoothing window.
    warm_ref = np.array([1533.8219, 1380.4337, 1242.4633, 1104.6660], dtype=np.float64)
    logp_grid = np.log10(p_grid)
    solar_log = np.log10(
        np.array([ML_SOLAR_ABUNDANCES[key] for key in ("He_H", "C_H", "O_H", "N_H", "S_H")], dtype=np.float64)
    )
    counts = {
        "sampled_test_runs": int(len(sampled)),
        "cold_top_count": 0,
        "cold_top_solar_log_rms_dex_le_0p3": 0,
        "warmshape_like_count": 0,
        "warmshape_like_solar_log_rms_dex_le_0p2": 0,
    }
    nearest_examples = []

    with h5py.File(raw_h5_path, "r") as handle:
        for run_id in sampled:
            group = handle[run_id]
            pressure_bar = np.asarray(group["inputs/pressure_bar"], dtype=np.float64)
            temperature_k = np.asarray(group["inputs/temperature_k"], dtype=np.float64)
            globals_log = np.log10(
                np.array(
                    [
                        float(np.asarray(group[f"globals/{key}"]))
                        for key in ("He_H", "C_H", "O_H", "N_H", "S_H")
                    ],
                    dtype=np.float64,
                )
            )
            globals_rms_dex = float(np.sqrt(np.mean((globals_log - solar_log) ** 2)))

            cold_top = bool(
                (temperature_k[0] >= 1700.0)
                and (temperature_k[-1] <= 280.0)
                and (pressure_bar[-1] <= 3.0e-7)
            )
            if cold_top:
                counts["cold_top_count"] += 1
                if globals_rms_dex <= 0.3:
                    counts["cold_top_solar_log_rms_dex_le_0p3"] += 1

            if pressure_bar.min() > p_grid.min() or pressure_bar.max() < p_grid.max():
                continue

            temperature_grid = np.interp(
                logp_grid,
                np.log10(pressure_bar[::-1]),
                temperature_k[::-1],
            )
            warmshape_like = bool(
                (abs(temperature_grid[0] - warm_ref[0]) <= 250.0)
                and (abs(temperature_grid[1] - warm_ref[1]) <= 200.0)
                and (abs(temperature_grid[2] - warm_ref[2]) <= 150.0)
                and (abs(temperature_grid[3] - warm_ref[3]) <= 150.0)
            )
            if warmshape_like:
                counts["warmshape_like_count"] += 1
                if globals_rms_dex <= 0.2:
                    counts["warmshape_like_solar_log_rms_dex_le_0p2"] += 1

            pt_score = float(
                np.sqrt(
                    np.mean(
                        ((temperature_grid - warm_ref) / np.array([250.0, 200.0, 150.0, 150.0])) ** 2
                    )
                )
            )
            nearest_examples.append(
                {
                    "run_id": str(run_id),
                    "score": pt_score + 0.5 * globals_rms_dex,
                    "globals_log_rms_dex": globals_rms_dex,
                    "bottom_temperature_k": float(temperature_k[0]),
                    "top_temperature_k": float(temperature_k[-1]),
                    "top_pressure_bar": float(pressure_bar[-1]),
                    "temperature_grid_k": [float(value) for value in temperature_grid],
                }
            )

    nearest_examples.sort(key=lambda item: item["score"])
    counts["nearest_examples"] = nearest_examples[:5]
    return counts


def _count_monitor_fail_rows(debug_dir):
    monitor_path = debug_dir / "monitor_output.dat"
    if not monitor_path.exists():
        return {"count": 0, "rows": []}

    fail_rows = []
    for line in monitor_path.read_text(encoding="utf-8").splitlines()[1:]:
        parts = line.split()
        if not parts:
            continue
        if any(status == "fail" for status in parts[8:]):
            fail_rows.append(int(parts[0]))
    return {"count": int(len(fail_rows)), "rows": fail_rows[:20]}


def _sample_live_fastchem_reproducibility(sample_limit=20):
    """Check whether the current live FastChem runtime reproduces stored raw runs."""

    run_ids_path = PROJECT_ROOT / "data" / MODEL / "processed" / "test" / "run_ids.json"
    raw_h5_path = PROJECT_ROOT / "data" / MODEL / "raw" / "runs.h5"
    if not run_ids_path.exists() or not raw_h5_path.exists():
        return {"error": f"Missing reproducibility inputs under {run_ids_path.parent}."}

    run_ids = json.loads(run_ids_path.read_text())
    stride = max(1, len(run_ids) // sample_limit)
    sampled = run_ids[::stride][:sample_limit]
    rows = []

    with h5py.File(raw_h5_path, "r") as handle:
        for run_id in sampled:
            group = handle[run_id]
            pressure_bar = np.asarray(group["inputs/pressure_bar"], dtype=np.float64)
            temperature_k = np.asarray(group["inputs/temperature_k"], dtype=np.float64)
            output_species = [
                value.decode("utf-8") if isinstance(value, bytes) else str(value)
                for value in np.asarray(group["inputs/output_species"])
            ]
            stored = np.asarray(group["equilibrium/ymix"], dtype=np.float64)
            globals_map = {
                key: float(np.asarray(group[f"globals/{key}"]))
                for key in ("He_H", "C_H", "O_H", "N_H", "S_H")
            }
            source_value = np.asarray(group["metadata/temperature_profile_source"]).item()
            source = source_value.decode("utf-8") if isinstance(source_value, bytes) else str(source_value)
            if source == "analytic":
                adjusted = bool(np.asarray(group["metadata/temperature_profile_analytic_convective_adjustment_applied"]))
                bucket = "analytic_convective" if adjusted else "analytic_radiative"
            else:
                bucket = source

            live = run_fastchem_online(
                FASTCHEM_SOURCE_ROOT,
                pressure_bar,
                temperature_k,
                globals_map,
                output_species,
                bundle.config,
            )
            mean_abs_dex = float(mean_abs_log10_error(stored, live))
            fail_rows = {"count": 0, "rows": []}
            debug_path = None
            if mean_abs_dex > 0.1:
                debug_path = DEBUG_DIR / "repro_sample" / str(run_id)
                _ = run_fastchem_online(
                    FASTCHEM_SOURCE_ROOT,
                    pressure_bar,
                    temperature_k,
                    globals_map,
                    output_species,
                    bundle.config,
                    debug_dir=debug_path,
                )
                fail_rows = _count_monitor_fail_rows(debug_path)

            rows.append(
                {
                    "run_id": str(run_id),
                    "bucket": bucket,
                    "mean_abs_dex": mean_abs_dex,
                    "fail_row_count": int(fail_rows["count"]),
                    "fail_rows_head": fail_rows["rows"],
                    "top_pressure_bar": float(pressure_bar[-1]),
                    "top_temperature_k": float(temperature_k[-1]),
                    "debug_dir": str(debug_path.resolve()) if debug_path is not None else None,
                }
            )

    rows.sort(key=lambda item: item["mean_abs_dex"], reverse=True)
    return {
        "sampled_test_runs": int(len(rows)),
        "count_mean_abs_dex_gt_0p1": int(sum(item["mean_abs_dex"] > 0.1 for item in rows)),
        "count_monitor_fail_rows_gt_0": int(sum(item["fail_row_count"] > 0 for item in rows)),
        "worst_runs": rows[:5],
    }


summary = {
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "bundle_path": str(BUNDLE_PATH.resolve()),
    "model": MODEL,
    "exogibbs_element_mode": EXOGIBBS_ELEMENT_MODE,
    "state_floor": float(STATE_FLOOR),
    "globals_map": {key: float(value) for key, value in ML_SOLAR_ABUNDANCES.items()},
    "species_labels": list(species_labels),
    "profiles": {},
}
npz_payload = {}

for result in profile_results:
    label = result["label"]
    slug = _slugify(label)
    p_ml = np.asarray(result["ml"], dtype=np.float64)
    p_fc = np.asarray(result["fastchem"], dtype=np.float64)
    p_eg = np.asarray(result["exogibbs"], dtype=np.float64)
    lr_ml_fc = np.asarray(result["log10_ratio_ml_fc"], dtype=np.float64)
    lr_fc_eg = np.asarray(result["log10_ratio_fc_eg"], dtype=np.float64)
    lr_ml_eg = np.asarray(result["log10_ratio_ml_eg"], dtype=np.float64)

    npz_payload[f"{slug}/pressure_bar"] = np.asarray(result["pressure_bar"], dtype=np.float64)
    npz_payload[f"{slug}/temperature_k"] = np.asarray(result["temperature_k"], dtype=np.float64)
    npz_payload[f"{slug}/ml"] = p_ml
    npz_payload[f"{slug}/fastchem"] = p_fc
    npz_payload[f"{slug}/exogibbs"] = p_eg
    npz_payload[f"{slug}/log10_ratio_ml_fc"] = lr_ml_fc
    npz_payload[f"{slug}/log10_ratio_fc_eg"] = lr_fc_eg
    npz_payload[f"{slug}/log10_ratio_ml_eg"] = lr_ml_eg

    summary["profiles"][label] = {
        "timings_ms": {key: float(value) for key, value in result["timings_ms"].items()},
        "fastchem_debug_dir": str(result["fastchem_debug_dir"]),
        "pairwise_mean_abs_dex": {
            "ml_vs_fastchem": float(mean_abs_log10_error(p_ml, p_fc)),
            "fastchem_vs_exogibbs": float(mean_abs_log10_error(p_fc, p_eg)),
            "ml_vs_exogibbs": float(mean_abs_log10_error(p_ml, p_eg)),
        },
        "pairwise_per_species": {
            "ml_vs_fastchem": _per_species_ratio_stats(lr_ml_fc),
            "fastchem_vs_exogibbs": _per_species_ratio_stats(lr_fc_eg),
            "ml_vs_exogibbs": _per_species_ratio_stats(lr_ml_eg),
        },
        "worst_species": {
            "ml_vs_fastchem": _top_species(lr_ml_fc, result["pressure_bar"], result["temperature_k"]),
            "fastchem_vs_exogibbs": _top_species(lr_fc_eg, result["pressure_bar"], result["temperature_k"]),
            "ml_vs_exogibbs": _top_species(lr_ml_eg, result["pressure_bar"], result["temperature_k"]),
        },
        "below_floor_counts": {
            "ml": _below_floor_counts(p_ml),
            "fastchem": _below_floor_counts(p_fc),
            "exogibbs": _below_floor_counts(p_eg),
        },
    }

summary["profiles"]["Warm Guillot"]["coverage_audit"] = _sample_warm_corner_coverage()
summary["live_fastchem_reproducibility_audit"] = _sample_live_fastchem_reproducibility()
(DEBUG_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
np.savez_compressed(DEBUG_DIR / "profiles.npz", **npz_payload)

warm = summary["profiles"]["Warm Guillot"]
audit = warm["coverage_audit"]
print(f"Saved debug summary to {DEBUG_DIR / 'summary.json'}")
print(f"Saved profile arrays to {DEBUG_DIR / 'profiles.npz'}")
print(
    "Warm Guillot: "
    f"ML-FC={warm['pairwise_mean_abs_dex']['ml_vs_fastchem']:.4f} dex, "
    f"FC-EG={warm['pairwise_mean_abs_dex']['fastchem_vs_exogibbs']:.4f} dex, "
    f"ML-EG={warm['pairwise_mean_abs_dex']['ml_vs_exogibbs']:.4f} dex"
)
print(
    "Warm Guillot CH4 below state_floor: "
    f"FastChem={warm['below_floor_counts']['fastchem']['CH4']['below_state_floor']} levels, "
    f"ExoGibbs={warm['below_floor_counts']['exogibbs']['CH4']['below_state_floor']} levels, "
    f"ML={warm['below_floor_counts']['ml']['CH4']['below_state_floor']} levels"
)
if 'error' not in audit:
    print(
        "Warm Guillot coverage audit: "
        f"warm-shape-like={audit['warmshape_like_count']} / {audit['sampled_test_runs']} sampled test runs, "
        f"warm-shape-like+solar-close={audit['warmshape_like_solar_log_rms_dex_le_0p2']} / {audit['sampled_test_runs']}"
    )
repro = summary["live_fastchem_reproducibility_audit"]
if 'error' not in repro:
    print(
        "Stored-vs-live FastChem audit: "
        f">0.1 dex mismatches={repro['count_mean_abs_dex_gt_0p1']} / {repro['sampled_test_runs']}, "
        f"monitor fail rows={repro['count_monitor_fail_rows_gt_0']} / {repro['sampled_test_runs']}"
    )


